# Greath North Run data processing

In [59]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as mplt
import seaborn as sns

In [60]:
raw_df = pd.read_excel("2024 Data Challenge Raw Data.xlsx")
raw_df.shape

(14614, 17)

## Row descriptions
|Variable |	Definition |
| :-: | :- |
| club |	The club the runner is a part of 
|position |	The position the runner finished in 
|time |	The time the runner took to finish the race
|id |	The runners id
|gender	| The runners gender
|trained_10_week |	The average number of times per week the runner trained 10 or more weeks ago
|trained_im	| The average number of times per week the runner trained less than e weeks ago
|has_trainer |	Whether the runner has a trainer or not
|cadence |	The average number of steps the runner took per minute
|age |	The age of the runner
|bmi |	The bmi of the runner
|n_marathons_run |	The number of marathons the runner has run before the race
|bib_colour |	The runners bib colour
|VO2_max |	The VO2 max of the runner
|heart_rate	| The resting heart rate of the runner
|shoe_size	| The runner's shoe size


NOTE:  
*Bib colours have meaning:  
Orange: 

In [61]:
raw_df.head()

#figure out unnamed column name- external sources?
#fix club names for all the records, fill up until you find text

,Unnamed: 0,club,position,time,id,gender,trained_10_week,trained_im,has_trainer,cadence,age,bmi,n_marathons_run,bib_colour,VO2_max,heart_rate,shoe_size
0,7448,NaN,14589,03:45:46,7426,female,0.789725,0.789725,0,163,36.0,999.00,0.0,green,12.718323,103.67,6
1,5363,NaN,14588,03:26:28,5345,female,1.670838,1.670838,0,157,30.0,19.05,0.0,green,12.718323,99.33,5
2,11420,NaN,14587,03:26:22,11387,female,1.917839,1.917839,0,165,38.0,21.13,0.0,green,13.460000,98.43,6
3,9408,Gump's training,14586,03:21:13,9384,male,0.000000,0.000000,0,154,38.0,18.86,0.0,green,20.010000,100.91,7
4,6604,NaN,14585,03:20:19,6585,female,0.000000,0.710092,0,166,43.0,19.75,0.0,green,12.718323,102.45,6


In [62]:
#no of missing values in each column 
raw_df.isnull().sum()

#autofill club information for all the other missing rows, boundaries to be defined from the excel spreadsheet 
#strategy for rows with missing time, impute/discard?
#

Unnamed: 0             0
club               11425
position               0
time                  47
id                     0
gender                 0
trained_10_week       74
trained_im           151
has_trainer            0
cadence                0
age                   51
bmi                    0
n_marathons_run       37
bib_colour             0
VO2_max              572
heart_rate           550
shoe_size              0
dtype: int64

In [63]:
raw_df.describe()

#trained 10 weeks, verify -999
#max BMI 999, fix that
#fix -9 in min for n_marathons_run
#fix max for n_marathons_run. is it normal for someone to run 1mil marathons? anomaly- discard

,Unnamed: 0,position,id,trained_10_week,trained_im,has_trainer,cadence,age,bmi,n_marathons_run,VO2_max,heart_rate,shoe_size
count,14614.000000,14614.000000,14614.000000,14540.000000,14463.000000,14614.000000,14614.000000,14563.000000,14614.000000,14577.000000,14042.000000,14064.000000,14614.000000
mean,7328.758656,7280.848159,7306.500000,-3.083608,-8.237465,0.078692,159.093540,36.102383,60.647316,3084.041984,44.585019,81.290607,7.656631
std,4231.313944,4219.205169,4218.842752,75.482450,106.119666,0.269266,32.010322,9.891467,189.231907,54865.742122,19.961066,10.775136,3.041907
min,0.000000,1.000000,0.000000,-999.000000,-999.000000,0.000000,1.000000,18.000000,15.050000,-9.000000,10.010000,58.680000,3.000000
25%,3665.250000,3627.250000,3653.250000,1.183613,2.008240,0.000000,161.000000,30.000000,21.120000,3.000000,28.260000,72.207500,5.000000
50%,7328.500000,7280.500000,7306.500000,2.745488,3.251724,0.000000,165.000000,36.000000,22.610000,6.000000,43.280000,80.740000,7.000000
75%,10992.750000,10933.750000,10959.750000,4.023773,4.332161,0.000000,169.000000,42.000000,24.180000,11.000000,60.827500,89.952500,10.000000
max,14660.000000,14589.000000,14613.000000,6.919653,6.919653,1.000000,189.000000,83.000000,999.000000,1000000.000000,95.850000,114.830000,13.000000


In [64]:

print(raw_df.bib_colour.value_counts())

bib_colour
green     5785
blue      4397
yellow    2925
red       1507
Name: count, dtype: int64


In [ ]:
# Cleaning steps
# [x] remove non existant: position, id, final time
# [x] trained_10_week & trained_im: replace negative with zero
# [x] has_trainer: one hot (already one hot)
# [x] bib colour: one hot
# [x] bmi: remove outliers
# [x] n_runs: remove negative and outliers
# [x] sex: one hot

df = raw_df
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)
    # Source - https://stackoverflow.com/a/49554761
    # Posted by Adil Warsi, modified by community. See post 'Timeline' for change history
    # Retrieved 2026-09-22, License - CC BY-SA 4.0
    df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)
df.dropna(subset=['time'], inplace=True)
df['time'] = pd.to_timedelta(df['time']).dt.total_seconds().astype(int)

# clean missing and negative values
df['trained_10_week'] = df['trained_10_week'].mask(df['trained_10_week'] < 0, 0)
df['trained_10_week'] = df['trained_10_week'].mask(df['trained_10_week'].isna(), 0)
df['trained_im'] = df['trained_im'].mask(df['trained_im'] < 0, 0)
df['trained_im'] = df['trained_im'].mask(df['trained_im'].isna(), 0)

# missing, negative and outliers
df['n_marathons_run'] = df['n_marathons_run'].mask(df['n_marathons_run'] < 0, 0)
df['n_marathons_run'] = df['n_marathons_run'].mask(df['n_marathons_run'].isna(), 0)
# todo: find and remove outliers

# multi hot encoding
pd.factorize(df['bib_colour'])
pd.factorize(df['gender'])

bmi_outliers = df['bmi'] == 999.0
print(bmi_outliers.value_counts())  # about 568 such cases so there is data error
df['bmi'] = df['bmi'].mask(df['bmi'] == 999.0, df['bmi'].mean())

# drop missing values cause they have strong correlation
df.dropna(subset=['VO2_max', 'heart_rate'])

df.describe()

bmi
False    14567
Name: count, dtype: int64


,position,time,trained_10_week,trained_im,has_trainer,cadence,age,bmi,n_marathons_run,VO2_max,heart_rate,shoe_size
count,14567.000000,14567.0,14567.000000,14567.000000,14567.000000,14567.000000,14516.000000,14567.000000,14567.000000,13998.000000,14019.000000,14567.000000
mean,7272.986545,0.0,2.607571,3.060508,0.078946,159.136747,36.105263,23.980303,3086.170660,44.612217,81.272823,7.659504
std,4218.630093,0.0,1.701968,1.539159,0.269663,31.925886,9.893728,7.648680,54884.512209,19.961066,10.775234,3.041941
min,1.000000,0.0,0.000000,0.000000,0.000000,1.000000,18.000000,15.050000,0.000000,10.010000,58.680000,3.000000
25%,3621.500000,0.0,1.157035,1.966382,0.000000,161.000000,30.000000,21.130000,3.000000,28.280000,72.190000,5.000000
50%,7264.000000,0.0,2.735884,3.233925,0.000000,165.000000,36.000000,22.610000,6.000000,43.330000,80.710000,7.000000
75%,10925.500000,0.0,4.017421,4.319452,0.000000,169.000000,42.000000,24.180000,11.000000,60.850000,89.940000,10.000000
max,14589.000000,0.0,6.919653,6.919653,1.000000,189.000000,83.000000,60.571726,1000000.000000,95.850000,114.830000,13.000000


In [66]:
df.info()

<class 'pandas.DataFrame'>
Index: 14567 entries, 0 to 14613
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   club             3182 non-null   str    
 1   position         14567 non-null  int64  
 2   time             14567 non-null  int64  
 3   gender           14567 non-null  str    
 4   trained_10_week  14567 non-null  float64
 5   trained_im       14567 non-null  float64
 6   has_trainer      14567 non-null  int64  
 7   cadence          14567 non-null  int64  
 8   age              14516 non-null  float64
 9   bmi              14567 non-null  float64
 10  n_marathons_run  14567 non-null  float64
 11  bib_colour       14567 non-null  str    
 12  VO2_max          13998 non-null  float64
 13  heart_rate       14019 non-null  float64
 14  shoe_size        14567 non-null  int64  
dtypes: float64(7), int64(5), str(3)
memory usage: 1.8 MB


In [67]:
test_df = df.dropna(subset=['VO2_max', 'heart_rate'])
test_df.shape
correlation_matrix = df[['VO2_max', 'heart_rate', 'time']].corr(method='pearson')

# 2. Print the results
print("--- Pearson Correlation Matrix ---")
print(correlation_matrix)

"""--- Pearson Correlation Matrix ---
             VO2_max  heart_rate      time
VO2_max     1.000000   -0.886878 -0.873315
heart_rate -0.886878    1.000000  0.920346
time       -0.873315    0.920346  1.000000"""



--- Pearson Correlation Matrix ---
             VO2_max  heart_rate      time
VO2_max     1.000000   -0.886878 -0.873315
heart_rate -0.886878    1.000000  0.920346
time       -0.873315    0.920346  1.000000


'--- Pearson Correlation Matrix ---\n             VO2_max  heart_rate      time\nVO2_max     1.000000   -0.886878 -0.873315\nheart_rate -0.886878    1.000000  0.920346\ntime       -0.873315    0.920346  1.000000'

#Since there is a very strong correlation between the VO2_max and heart_rate, populating it with an average will affect the reliabilioty of the model to accurately predict the performance of a runner, and since the unique number of rows with the intersection of both these values is less than 10% of the data, we have chosen to eliminate it


In [68]:
(df["bib_colour"][df["cadence"] == 1.00000]).value_counts()

bib_colour
green     241
blue      156
yellow    100
red        55
Name: count, dtype: int64

In [69]:

average_timings = df.groupby(['bib_colour', 'gender'])['time'].mean().reset_index()
average_timings.columns = ['Bib Color', 'Gender', 'Average Seconds']

# 2. Convert the average seconds into HH:MM:SS format
average_timings['Average HH:MM:SS'] = pd.to_timedelta(average_timings['Average Seconds'].round(), unit='s')
average_timings['Average HH:MM:SS'] = average_timings['Average HH:MM:SS'].apply(lambda x: str(x).split()[-1])

print(average_timings)

  Bib Color  Gender  Average Seconds Average HH:MM:SS
0      blue  female      5677.765200         01:34:38
1      blue    male      5596.142252         01:33:16
2     green  female      7110.314149         01:58:30
3     green    male      6985.460809         01:56:25
4       red  female      4120.236220         01:08:40
5       red    male      4094.046326         01:08:14
6    yellow  female      4901.890724         01:21:42
7    yellow    male      4823.842475         01:20:24


In [70]:
# a.
# - clean and why you clean what you clean
# - mean, std, min, max, basically .describe()
# - yap about distribution (maybe pair plot)

# b.
# - pearson correlation matrix

# c.
# 